# Data Prepartion 


<h3>importing pakages</h3>

In [1]:
# importing pakages 
import os
import glob
import pandas as pd
import numpy as np


<h3>Load Raw Data</h3>

In [2]:
# path of data
raw_data_path = '../data/raw/1429_1.csv'

# Check if file exists
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"Raw dataset not found at {raw_data_path}. Please download and place it in data/raw/")

# Load raw dataset
df_raw = pd.read_csv(raw_data_path)
print(f"Raw Data Loaded Successfully! Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")


Raw Data Loaded Successfully! Shape: 34660 rows, 21 columns.


C:\Users\shubh\AppData\Local\Temp\ipykernel_12348\3648518767.py:9: DtypeWarning: Columns (1: name, 10: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(raw_data_path)


<h3>Standardize Column Names</h3>

In [3]:
# Convert column names to snake_case

df_clean = df_raw.copy()

# Map columns standard names
column_mapping = {
    'id': 'product_id',
    'name': 'product_name',
    'brand': 'brand',
    'categories': 'category',
    'reviews.rating': 'rating',
    'reviews.text': 'review_text',
    'reviews.title': 'title',
    'reviews.doRecommend': 'recommended_raw',
    'reviews.numHelpful': 'helpful_votes'
}

df_clean = df_clean.rename(columns=column_mapping)

# Retain required fields
required_cols = ['product_id', 'product_name', 'brand', 'category', 'rating', 'review_text', 'title', 'recommended_raw', 'helpful_votes']
df_clean = df_clean[[col for col in required_cols if col in df_clean.columns]]

print("Standardized Column Names:")
print(list(df_clean.columns))

Standardized Column Names:
['product_id', 'product_name', 'brand', 'category', 'rating', 'review_text', 'title', 'recommended_raw', 'helpful_votes']


<h3>Handle Missing Values</h3>

In [4]:
print("Missing values before cleaning:")
print(df_clean.isnull().sum())
print(f"\nTotal rows: {len(df_clean):,}")

Missing values before cleaning:
product_id            0
product_name       6760
brand                 0
category              0
rating               33
review_text           1
title                 6
recommended_raw     594
helpful_votes       529
dtype: int64

Total rows: 34,660


In [5]:
# Substitute missing text fields instead of dropping records
df_clean['review_text'] = df_clean['review_text'].fillna('Missing')
df_clean['title'] = df_clean['title'].fillna('No Title')
df_clean['brand'] = df_clean['brand'].fillna('Amazon')
df_clean['category'] = df_clean['category'].fillna('Electronics')
df_clean['helpful_votes'] = df_clean['helpful_votes'].fillna(0)

# Ensure numeric ratings are clean
df_clean = df_clean.dropna(subset=['rating'])
df_clean['rating'] = df_clean['rating'].astype(int)

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())
print(f"\nTotal retained rows: {len(df_clean):,}")


Missing values after cleaning:
product_id            0
product_name       6759
brand                 0
category              0
rating                0
review_text           0
title                 0
recommended_raw     561
helpful_votes         0
dtype: int64

Total retained rows: 34,627


<h3>Variable Derivation</h3>

In [6]:
# Extract primary high-level category
df_clean['category'] = df_clean['category'].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else 'Electronics'
)

In [7]:
print(df_clean['recommended_raw'].unique())
print(df_clean['rating'].unique())

[True False nan]
[5 4 2 1 3]


In [9]:
# Binary Variable (recommended_indicator: 1 = Recommended, 0 = Not Recommended)

# Convert string boolean variations ('true'/'false') to 1/0
raw_map = {'true': 1, 'false': 0}
df_clean['recommended_ind'] = df_clean['recommended_raw'].astype(str).str.lower().map(raw_map)

# Impute missing values based on rating (Rating >= 4 is Recommended)
rating_good = (df_clean['rating'] >= 4).astype(int)
df_clean['recommended_ind'] = df_clean['recommended_ind'].fillna(rating_good).astype(int)


# String label for visualizations
df_clean['recommended_label'] = df_clean['recommended_ind'].map({1: 'Recommended', 0: 'Not Recommended'})

# 2. Text Metrics 
df_clean['review_char_count'] = df_clean['review_text'].apply(lambda x: 0 if x == 'Missing' else len(str(x)))
df_clean['review_word_count'] = df_clean['review_text'].apply(lambda x: 0 if x == 'Missing' else len(str(x).split()))
df_clean['has_title'] = df_clean['title'].apply(lambda x: 0 if x == 'No Title' else 1)

print("Variable Derivation Sample:")
df_clean[['review_text', 'rating', 'recommended_ind','recommended_label', 'review_word_count','review_char_count']].head()

Variable Derivation Sample:


,review_text,rating,recommended_ind,recommended_label,review_word_count,review_char_count
0,This product so far has not disappointed. My c...,5,1,Recommended,27,143
1,great for beginner or experienced person. Boug...,5,1,Recommended,14,75
2,Inexpensive tablet for him to use and learn on...,5,1,Recommended,26,131
3,I've had my Fire HD 8 two weeks now and I love...,4,1,Recommended,117,593
4,I bought this for my grand daughter when she c...,5,1,Recommended,117,613


<h3>Export Cleaned Dataset</h3>

In [10]:
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/cleaned_reviews.csv'

df_clean.to_csv(output_path, index=False)
print(f"SUCCESS: Amazon Consumer Reviews dataset processed and saved to {output_path}")
print(f"Final Processed Shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns.")

SUCCESS: Amazon Consumer Reviews dataset processed and saved to ../data/processed/cleaned_reviews.csv
Final Processed Shape: 34627 rows, 14 columns.
